# Whale Alert & Price Movement Correlation Analysis

Analyze whether whale transactions predict price movements.

**Questions we're answering:**
1. Do exchange inflows (selling pressure) precede price drops?
2. Do exchange outflows (accumulation) precede price rises?
3. How much lead time do whale signals provide?
4. Which coins show strongest whale-price correlation?

**Data Source:** `news_signals` table where `source='whale_alert'`

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re

# Style settings
plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

# Connect to snapshot database
DB_PATH = '../data/pythia_snapshot.duckdb'

try:
    conn = duckdb.connect(DB_PATH, read_only=True)
    print('Connected to DuckDB snapshot')
except Exception as e:
    print(f'Error: {e}')
    print('\nMake sure pythia_snapshot.duckdb exists')

## 1. Data Overview

In [ ]:
# Check what data we have
print('=== TABLE COUNTS ===')
for table in ['news_signals', 'ohlcv', 'features']:
    try:
        count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        print(f'{table}: {count:,} rows')
    except Exception as e:
        print(f'{table}: {e}')

print('\n=== WHALE ALERTS IN news_signals ===')
result = conn.execute('''
    SELECT COUNT(*) as count,
           MIN(timestamp) as first, 
           MAX(timestamp) as last,
           COUNT(DISTINCT symbol) as symbols
    FROM news_signals 
    WHERE source = 'whale_alert'
''').fetchone()
print(f'Count: {result[0]}')
print(f'From: {result[1]}')
print(f'To: {result[2]}')
print(f'Symbols: {result[3]}')

In [ ]:
# Load whale alerts and parse the title for details
whale_df = conn.execute('''
    SELECT timestamp, symbol, title, confidence, event_type
    FROM news_signals 
    WHERE source = 'whale_alert'
    ORDER BY timestamp
''').df()

print(f'Loaded {len(whale_df)} whale alerts')
print('\nSample titles:')
for t in whale_df['title'].head(10):
    print(f'  {t}')

In [ ]:
# Parse whale alert titles to extract amount and direction
def parse_whale_title(title):
    """Extract amount_usd and flow direction from whale alert title."""
    result = {'amount_usd': 0, 'direction': 'unknown', 'from_name': '', 'to_name': ''}
    
    # Extract USD amount - look for ($X,XXX,XXX) or ($X.XM) patterns
    usd_match = re.search(r'\$([\d,]+(?:\.\d+)?(?:[MBK])?)', title)
    if usd_match:
        amount_str = usd_match.group(1).replace(',', '')
        multiplier = 1
        if amount_str.endswith('B'):
            multiplier = 1e9
            amount_str = amount_str[:-1]
        elif amount_str.endswith('M'):
            multiplier = 1e6
            amount_str = amount_str[:-1]
        elif amount_str.endswith('K'):
            multiplier = 1e3
            amount_str = amount_str[:-1]
        try:
            result['amount_usd'] = float(amount_str) * multiplier
        except:
            pass
    
    # Extract from/to if present (format: "from → to" or "from -> to")
    arrow_match = re.search(r'\|\s*(.+?)\s*[→->]+\s*(.+?)$', title)
    if arrow_match:
        result['from_name'] = arrow_match.group(1).strip()
        result['to_name'] = arrow_match.group(2).strip()
        
        # Determine direction based on from/to
        exchanges = ['binance', 'coinbase', 'kraken', 'okx', 'bybit', 'huobi', 'kucoin', 'bitfinex', 'gemini']
        from_lower = result['from_name'].lower()
        to_lower = result['to_name'].lower()
        
        from_is_exchange = any(ex in from_lower for ex in exchanges)
        to_is_exchange = any(ex in to_lower for ex in exchanges)
        
        if to_is_exchange and not from_is_exchange:
            result['direction'] = 'exchange_inflow'  # Moving TO exchange = selling pressure
        elif from_is_exchange and not to_is_exchange:
            result['direction'] = 'exchange_outflow'  # Moving FROM exchange = accumulation
        else:
            result['direction'] = 'transfer'
    else:
        result['direction'] = 'transfer'
    
    return result

# Apply parsing
parsed = whale_df['title'].apply(parse_whale_title).apply(pd.Series)
whale_df = pd.concat([whale_df, parsed], axis=1)

print('=== PARSED WHALE DATA ===')
print(f"Direction counts:")
print(whale_df['direction'].value_counts())
print(f"\nTotal USD volume: ${whale_df['amount_usd'].sum():,.0f}")
print(f"Average transaction: ${whale_df['amount_usd'].mean():,.0f}")

In [ ]:
# Visualize whale transaction types
direction_counts = whale_df['direction'].value_counts()
direction_volumes = whale_df.groupby('direction')['amount_usd'].sum() / 1e9

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(direction_counts.index, direction_counts.values, color=['red', 'green', 'gray', 'blue'][:len(direction_counts)])
axes[0].set_title('Transaction Count by Type')
axes[0].set_xlabel('Direction')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(direction_volumes.index, direction_volumes.values, color=['red', 'green', 'gray', 'blue'][:len(direction_volumes)])
axes[1].set_title('Total Volume by Type ($B)')
axes[1].set_xlabel('Direction')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Whale Events → Price Impact Analysis

In [ ]:
# Get OHLCV data for joining
ohlcv_df = conn.execute('''
    SELECT symbol, timestamp, close
    FROM ohlcv 
    WHERE timeframe = '1m'
    ORDER BY symbol, timestamp
''').df()

print(f'Loaded {len(ohlcv_df):,} OHLCV rows')
print(f'Symbols: {ohlcv_df["symbol"].nunique()}')
print(f'Date range: {ohlcv_df["timestamp"].min()} to {ohlcv_df["timestamp"].max()}')

In [ ]:
# Join whale events with price data at different time offsets
def get_price_at_offset(whale_row, ohlcv_df, offset_minutes):
    """Get price for symbol at whale_time + offset."""
    target_time = whale_row['timestamp'] + pd.Timedelta(minutes=offset_minutes)
    symbol_data = ohlcv_df[ohlcv_df['symbol'] == whale_row['symbol']]
    
    if len(symbol_data) == 0:
        return np.nan
    
    # Find closest timestamp
    idx = (symbol_data['timestamp'] - target_time).abs().idxmin()
    price_row = symbol_data.loc[idx]
    
    # Only use if within 2 minutes of target
    if abs((price_row['timestamp'] - target_time).total_seconds()) > 120:
        return np.nan
    
    return price_row['close']

# Filter to exchange inflows/outflows with significant amounts
whale_filtered = whale_df[
    (whale_df['direction'].isin(['exchange_inflow', 'exchange_outflow'])) &
    (whale_df['amount_usd'] >= 1_000_000)  # $1M+ only
].copy()

print(f'Filtered to {len(whale_filtered)} whale events ($1M+ inflows/outflows)')

# This is slow - let's use a vectorized approach with DuckDB
whale_filtered['timestamp_rounded'] = whale_filtered['timestamp'].dt.floor('min')

In [ ]:
# Use DuckDB for efficient price lookups
# Register the whale dataframe
conn.register('whale_events', whale_filtered)

whale_price_impact = conn.execute('''
    SELECT 
        w.timestamp as whale_time,
        w.symbol,
        w.amount_usd,
        w.direction,
        w.from_name,
        w.to_name,
        w.title,
        
        -- Price at different times
        o_at.close as price_at_whale,
        o_5m.close as price_5m_after,
        o_15m.close as price_15m_after,
        o_30m.close as price_30m_after,
        o_60m.close as price_60m_after,
        
        -- Calculate returns
        (o_5m.close - o_at.close) / NULLIF(o_at.close, 0) as return_5m,
        (o_15m.close - o_at.close) / NULLIF(o_at.close, 0) as return_15m,
        (o_30m.close - o_at.close) / NULLIF(o_at.close, 0) as return_30m,
        (o_60m.close - o_at.close) / NULLIF(o_at.close, 0) as return_60m
        
    FROM whale_events w
    LEFT JOIN ohlcv o_at ON w.symbol = o_at.symbol 
        AND o_at.timeframe = '1m'
        AND o_at.timestamp = date_trunc('minute', w.timestamp)
    LEFT JOIN ohlcv o_5m ON w.symbol = o_5m.symbol 
        AND o_5m.timeframe = '1m'
        AND o_5m.timestamp = date_trunc('minute', w.timestamp) + INTERVAL '5 minutes'
    LEFT JOIN ohlcv o_15m ON w.symbol = o_15m.symbol 
        AND o_15m.timeframe = '1m'
        AND o_15m.timestamp = date_trunc('minute', w.timestamp) + INTERVAL '15 minutes'
    LEFT JOIN ohlcv o_30m ON w.symbol = o_30m.symbol 
        AND o_30m.timeframe = '1m'
        AND o_30m.timestamp = date_trunc('minute', w.timestamp) + INTERVAL '30 minutes'
    LEFT JOIN ohlcv o_60m ON w.symbol = o_60m.symbol 
        AND o_60m.timeframe = '1m'
        AND o_60m.timestamp = date_trunc('minute', w.timestamp) + INTERVAL '60 minutes'
    WHERE o_at.close IS NOT NULL
''').df()

conn.unregister('whale_events')

print(f'Whale events with price data: {len(whale_price_impact):,}')
whale_price_impact.head(10)

In [ ]:
# Analyze: Do exchange inflows predict price drops?
inflows = whale_price_impact[whale_price_impact['direction'] == 'exchange_inflow'].copy()
outflows = whale_price_impact[whale_price_impact['direction'] == 'exchange_outflow'].copy()

print('=== EXCHANGE INFLOWS (Selling Pressure) ===')
print(f'Count: {len(inflows)}')
if len(inflows) > 0:
    for col in ['return_5m', 'return_15m', 'return_30m', 'return_60m']:
        data = inflows[col].dropna()
        if len(data) > 0:
            mean_ret = data.mean() * 100
            median_ret = data.median() * 100
            pct_negative = (data < 0).mean() * 100
            print(f'  {col}: mean={mean_ret:.3f}%, median={median_ret:.3f}%, {pct_negative:.1f}% negative')

print('\n=== EXCHANGE OUTFLOWS (Accumulation) ===')
print(f'Count: {len(outflows)}')
if len(outflows) > 0:
    for col in ['return_5m', 'return_15m', 'return_30m', 'return_60m']:
        data = outflows[col].dropna()
        if len(data) > 0:
            mean_ret = data.mean() * 100
            median_ret = data.median() * 100
            pct_positive = (data > 0).mean() * 100
            print(f'  {col}: mean={mean_ret:.3f}%, median={median_ret:.3f}%, {pct_positive:.1f}% positive')

In [ ]:
# Visualize: Return distributions after whale events
if len(inflows) > 0 or len(outflows) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    timeframes = ['return_5m', 'return_15m', 'return_30m', 'return_60m']
    titles = ['5 min after', '15 min after', '30 min after', '60 min after']

    for ax, tf, title in zip(axes.flat, timeframes, titles):
        # Filter outliers for better visualization
        inflow_data = inflows[tf].dropna() if len(inflows) > 0 else pd.Series()
        outflow_data = outflows[tf].dropna() if len(outflows) > 0 else pd.Series()
        
        inflow_data = inflow_data[np.abs(inflow_data) < 0.1] if len(inflow_data) > 0 else inflow_data
        outflow_data = outflow_data[np.abs(outflow_data) < 0.1] if len(outflow_data) > 0 else outflow_data
        
        if len(inflow_data) > 0:
            ax.hist(inflow_data * 100, bins=30, alpha=0.5, label=f'Inflow (n={len(inflow_data)})', color='red')
            ax.axvline(x=inflow_data.mean()*100, color='red', linestyle='-', alpha=0.8)
        if len(outflow_data) > 0:
            ax.hist(outflow_data * 100, bins=30, alpha=0.5, label=f'Outflow (n={len(outflow_data)})', color='green')
            ax.axvline(x=outflow_data.mean()*100, color='green', linestyle='-', alpha=0.8)
        
        ax.axvline(x=0, color='white', linestyle='--', alpha=0.5)
        ax.set_title(f'Price Change {title}')
        ax.set_xlabel('Return (%)')
        ax.legend(fontsize=8)

    plt.suptitle('Price Movement After Whale Transactions', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No inflow/outflow data to plot')

## 3. All Whale Moves Analysis

In [ ]:
# Look at ALL whale moves (not just identified inflows/outflows)
# Use the full whale_df and join with prices

whale_all = whale_df[whale_df['amount_usd'] >= 1_000_000].copy()
conn.register('whale_all', whale_all)

whale_all_prices = conn.execute('''
    SELECT 
        w.timestamp as whale_time,
        w.symbol,
        w.amount_usd,
        w.direction,
        w.title,
        
        o_at.close as price_at_whale,
        o_30m.close as price_30m_after,
        (o_30m.close - o_at.close) / NULLIF(o_at.close, 0) as return_30m
        
    FROM whale_all w
    LEFT JOIN ohlcv o_at ON w.symbol = o_at.symbol 
        AND o_at.timeframe = '1m'
        AND o_at.timestamp = date_trunc('minute', w.timestamp)
    LEFT JOIN ohlcv o_30m ON w.symbol = o_30m.symbol 
        AND o_30m.timeframe = '1m'
        AND o_30m.timestamp = date_trunc('minute', w.timestamp) + INTERVAL '30 minutes'
    WHERE o_at.close IS NOT NULL
''').df()

conn.unregister('whale_all')

print(f'All whale events with price data: {len(whale_all_prices):,}')
print(f'\nBy direction:')
print(whale_all_prices['direction'].value_counts())

In [ ]:
# Scatter: Whale size vs price impact (all moves)
fig, ax = plt.subplots(figsize=(12, 6))

data = whale_all_prices.dropna(subset=['return_30m'])
data = data[np.abs(data['return_30m']) < 0.1]  # Filter outliers

colors = data['direction'].map({
    'exchange_inflow': 'red',
    'exchange_outflow': 'green',
    'transfer': 'gray',
    'unknown': 'blue'
})

ax.scatter(data['amount_usd'] / 1e6, data['return_30m'] * 100, 
           alpha=0.6, s=30, c=colors)
ax.axhline(y=0, color='white', linestyle='--', alpha=0.5)
ax.set_xlabel('Transaction Amount ($M)')
ax.set_ylabel('30min Return (%)')
ax.set_title('Whale Transaction Size vs Price Impact (30min)')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Exchange Inflow'),
    Patch(facecolor='green', label='Exchange Outflow'),
    Patch(facecolor='gray', label='Transfer'),
]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

## 4. By Symbol Analysis

In [ ]:
# Which symbols have the most whale activity?
symbol_stats = whale_all_prices.groupby('symbol').agg({
    'amount_usd': ['count', 'sum', 'mean'],
    'return_30m': ['mean', 'std', 'count']
}).reset_index()

symbol_stats.columns = ['symbol', 'whale_count', 'total_usd', 'avg_usd', 'mean_return', 'std_return', 'return_count']
symbol_stats = symbol_stats[symbol_stats['whale_count'] >= 3]  # Min 3 events
symbol_stats = symbol_stats.sort_values('whale_count', ascending=False)

print('=== TOP SYMBOLS BY WHALE ACTIVITY ===')
print(symbol_stats.head(15).to_string())

In [ ]:
# Visualize top symbols
top_symbols = symbol_stats.head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_symbols['symbol'], top_symbols['whale_count'])
axes[0].set_xlabel('Number of Whale Events')
axes[0].set_title('Top Symbols by Whale Activity')
axes[0].invert_yaxis()

axes[1].barh(top_symbols['symbol'], top_symbols['mean_return'] * 100)
axes[1].axvline(x=0, color='white', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Mean 30min Return (%)')
axes[1].set_title('Price Impact After Whale Events')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Time Series: BTC Whale Activity vs Price

In [ ]:
# Focus on BTC
SYMBOL = 'BTC-USD'

btc_whales = whale_all_prices[whale_all_prices['symbol'] == SYMBOL].copy()
print(f'{SYMBOL} whale events: {len(btc_whales)}')

# Get BTC price series
btc_price = conn.execute(f'''
    SELECT timestamp, close
    FROM ohlcv
    WHERE symbol = '{SYMBOL}' AND timeframe = '1m'
    ORDER BY timestamp
''').df()

print(f'BTC price data: {len(btc_price)} rows')
print(f'Date range: {btc_price["timestamp"].min()} to {btc_price["timestamp"].max()}')

In [ ]:
# Plot BTC price with whale events overlaid
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Price chart
ax = axes[0]
ax.plot(btc_price['timestamp'], btc_price['close'], color='cyan', linewidth=0.5, alpha=0.8)

# Mark whale events
for _, row in btc_whales.iterrows():
    color = 'red' if row['direction'] == 'exchange_inflow' else 'green' if row['direction'] == 'exchange_outflow' else 'yellow'
    ax.axvline(x=row['whale_time'], color=color, alpha=0.3, linewidth=1)

ax.set_ylabel('Price ($)')
ax.set_title(f'{SYMBOL} Price with Whale Events (Red=Inflow, Green=Outflow, Yellow=Transfer)')

# Whale amounts
ax = axes[1]
colors = btc_whales['direction'].map({
    'exchange_inflow': 'red',
    'exchange_outflow': 'green',
    'transfer': 'yellow',
    'unknown': 'blue'
})
ax.bar(btc_whales['whale_time'], btc_whales['amount_usd'] / 1e6, 
       color=colors, alpha=0.7, width=0.01)
ax.set_ylabel('Amount ($M)')
ax.set_xlabel('Time')
ax.set_title('Whale Transaction Amounts')

plt.tight_layout()
plt.show()

## 6. Summary & Conclusions

In [ ]:
print('='*60)
print('WHALE ALERT CORRELATION ANALYSIS SUMMARY')
print('='*60)

print(f'''
DATA ANALYZED:
- Total whale alerts: {len(whale_df):,}
- Whale events with price data: {len(whale_all_prices):,}
- Exchange inflows: {len(inflows):,}
- Exchange outflows: {len(outflows):,}
- Date range: {whale_df['timestamp'].min()} to {whale_df['timestamp'].max()}
''')

# Inflow impact
if len(inflows) > 0:
    inflow_30m = inflows['return_30m'].dropna()
    if len(inflow_30m) > 0:
        mean_ret = inflow_30m.mean() * 100
        pct_negative = (inflow_30m < 0).mean() * 100
        print(f'EXCHANGE INFLOWS (selling pressure):')
        print(f'  - Count: {len(inflow_30m)}')
        print(f'  - Mean 30min return: {mean_ret:.3f}%')
        print(f'  - % followed by price drop: {pct_negative:.1f}%')
        print(f'  - Verdict: {"SOMEWHAT PREDICTIVE" if pct_negative > 55 else "NOT CLEARLY PREDICTIVE"}')

# Outflow impact  
if len(outflows) > 0:
    outflow_30m = outflows['return_30m'].dropna()
    if len(outflow_30m) > 0:
        mean_ret = outflow_30m.mean() * 100
        pct_positive = (outflow_30m > 0).mean() * 100
        print(f'\nEXCHANGE OUTFLOWS (accumulation):')
        print(f'  - Count: {len(outflow_30m)}')
        print(f'  - Mean 30min return: {mean_ret:.3f}%')
        print(f'  - % followed by price rise: {pct_positive:.1f}%')
        print(f'  - Verdict: {"SOMEWHAT PREDICTIVE" if pct_positive > 55 else "NOT CLEARLY PREDICTIVE"}')

print(f'''
NOTES:
- Only {len(whale_df)} whale alerts over ~2 days - need more data
- Direction parsing is heuristic (based on exchange names in title)
- Many alerts don't specify clear inflow/outflow direction
- Consider collecting data for 1-2 weeks for better statistical power

NEXT STEPS:
- Continue collecting whale data via whale_transactions table
- Re-run analysis after 1 week of data collection
- Focus on large moves ($10M+) for stronger signal
''')

In [ ]:
# Close connection
conn.close()
print('Database connection closed')